In [70]:
import numpy as np

# Configuration parameters for the (3,4)-regular LDPC base matrix
size = 42     # Example Submatrix size (Z value for WiMAX/WiFi sizes vary, 38 is typical for some max shift values here like 37)
M_b = 3       # Number of macro rows in base matrix
N_b = 16      # Number of macro columns in base matrix

# Shift values for the quasi-cyclic matrix (-1 means Bypass/All-Zero submatrix)
# Edited to ensure exactly 1 bypass per macro row and macro column.
shift_matrix = np.array([
    [ 29, 30,  0,  8, 33, 22, 17,  4, 27, 28, 20, 27, 24, 23, -1,  0],
    [ 37, 31, 18, 23, 11, 21,  6, 20, 32,  9, 12, 29, 10,  0, 13, -1],
    [ 25, 22,  4, 34, 31,  3, 14, 15,  4,  2, 14, 18, 13, -1, 22, 24]
])

M = M_b * size
N = N_b * size

# Generate the full parity-check matrix H
matrix_orig = np.zeros((M, N), dtype=int)

for i in range(M_b):
    for j in range(N_b):
        shift = shift_matrix[i, j]
        row_start = i * size
        col_start = j * size
        
        if shift == -1:
            submat = np.zeros((size, size), dtype=int)
        else:
            submat = np.roll(np.eye(size, dtype=int), shift, axis=1)
            
        matrix_orig[row_start:row_start+size, col_start:col_start+size] = submat

matrix = matrix_orig.copy()
print("Generated Full H Matrix:")
print(matrix_orig.shape)


Generated Full H Matrix:
(126, 672)


In [71]:
rows, cols = matrix_orig.shape
num_layers = rows // size

def prev_one_in_row(row_ones, col):
    before = [x for x in row_ones if x < col]
    if before:
        return max(before)
    else:
        return max(row_ones)

print("Bypass List (previous 1 to bypass):")
for j in range(cols):
    bypass_val = 15 # Default 4'hF
    for l in range(num_layers):
        sub_col = matrix_orig[l*size:(l+1)*size, j]
        if np.all(sub_col == 0):
            abs_row = (l * size) + (j % size)
            row_ones = np.where(matrix_orig[abs_row, :] == 1)[0].tolist()
            if row_ones:
                bypass_val = prev_one_in_row(row_ones, j)
            break
            
    print(f"        bypass_list[{j:2d}] = 4'h{bypass_val:X};")


Bypass List (previous 1 to bypass):
        bypass_list[ 0] = 4'hF;
        bypass_list[ 1] = 4'hF;
        bypass_list[ 2] = 4'hF;
        bypass_list[ 3] = 4'hF;
        bypass_list[ 4] = 4'hF;
        bypass_list[ 5] = 4'hF;
        bypass_list[ 6] = 4'hF;
        bypass_list[ 7] = 4'hF;
        bypass_list[ 8] = 4'hF;
        bypass_list[ 9] = 4'hF;
        bypass_list[10] = 4'hF;
        bypass_list[11] = 4'hF;
        bypass_list[12] = 4'hF;
        bypass_list[13] = 4'hF;
        bypass_list[14] = 4'hF;
        bypass_list[15] = 4'hF;
        bypass_list[16] = 4'hF;
        bypass_list[17] = 4'hF;
        bypass_list[18] = 4'hF;
        bypass_list[19] = 4'hF;
        bypass_list[20] = 4'hF;
        bypass_list[21] = 4'hF;
        bypass_list[22] = 4'hF;
        bypass_list[23] = 4'hF;
        bypass_list[24] = 4'hF;
        bypass_list[25] = 4'hF;
        bypass_list[26] = 4'hF;
        bypass_list[27] = 4'hF;
        bypass_list[28] = 4'hF;
        bypass_list[29] = 4'hF;
    

In [72]:
cn_pe_table = []
cn_degree = []

print("cn_pe_table (indices of 1s per CN, excluding bypasses):")
for r in range(rows):
    ones_idx = np.where(matrix_orig[r, :] == 1)[0].tolist()
    cn_pe_table.append(ones_idx)
    cn_degree.append(len(ones_idx))

    layer = r//size

    for i in range(len(cn_pe_table[r])):
        print(f"        cn_pe_table[{r}][{i}] = {ones_idx[i]};")
        print(f"        cn_layer_table[{r}][{i}] = {layer};\n")

    print(f"\n        degree[{r}] = {cn_degree[-1]};\n")


cn_pe_table (indices of 1s per CN, excluding bypasses):
        cn_pe_table[0][0] = 29;
        cn_layer_table[0][0] = 0;

        cn_pe_table[0][1] = 72;
        cn_layer_table[0][1] = 0;

        cn_pe_table[0][2] = 84;
        cn_layer_table[0][2] = 0;

        cn_pe_table[0][3] = 134;
        cn_layer_table[0][3] = 0;

        cn_pe_table[0][4] = 201;
        cn_layer_table[0][4] = 0;

        cn_pe_table[0][5] = 232;
        cn_layer_table[0][5] = 0;

        cn_pe_table[0][6] = 269;
        cn_layer_table[0][6] = 0;

        cn_pe_table[0][7] = 298;
        cn_layer_table[0][7] = 0;

        cn_pe_table[0][8] = 363;
        cn_layer_table[0][8] = 0;

        cn_pe_table[0][9] = 406;
        cn_layer_table[0][9] = 0;

        cn_pe_table[0][10] = 440;
        cn_layer_table[0][10] = 0;

        cn_pe_table[0][11] = 489;
        cn_layer_table[0][11] = 0;

        cn_pe_table[0][12] = 528;
        cn_layer_table[0][12] = 0;

        cn_pe_table[0][13] = 569;
        cn_layer_table[

In [73]:
import numpy as np

# Variable size of submatrix (layer = size)
size = 3

# Input matrix
# matrix = np.array([
#     [0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1],
#     [0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0],
#     [1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0],
#     [0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0],
#     [0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0],
#     [0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1],
#     [0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0],
#     [0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1],
#     [1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0]
# ])

rows, cols = matrix.shape
num_layers = rows // size

# Replace bypass (all-zero) submatrices with identity matrix
for i in range(0, rows, size):
    for j in range(0, cols, size):
        sub = matrix[i:i+size, j:j+size]
        if np.all(sub == 0):
            matrix[i:i+size, j:j+size] = np.eye(sub.shape[0], sub.shape[1])

def prev_one_in_row(row_ones, col):
    """Return the column index of the previous 1 in the row before 'col'.
    If none exists (col is leftmost 1), wrap around to the last 1 in the row."""
    before = [x for x in row_ones if x < col]
    if before:
        return max(before)   # nearest 1 to the left
    else:
        return max(row_ones) # wrap: last 1 in the row

# Generate edgelist:
# For each column, for each layer, find the column index of the previous 1
# in the same row as the current column's 1. Store as packed hex nibbles.
# Nibble[0] (MSN) = layer 0, Nibble[1] = layer 1, Nibble[2] = layer 2, ...
print("EdgeList (previous 1 per layer, packed as hex):")
for col in range(cols):
    nibbles = []
    for layer in range(num_layers):
        row_start = layer * size
        # Find which row in this layer has a 1 for this column
        rel_rows = np.where(matrix[row_start:row_start+size, col] == 1)[0]
        if len(rel_rows) == 0:
            nibbles.append(0)
            continue
        abs_row = row_start + rel_rows[0]
        # Get all columns in this row that have a 1
        row_ones = np.where(matrix[abs_row, :] == 1)[0].tolist()
        # Find the previous 1 in the row (with circular wrap)
        prev_idx = prev_one_in_row(row_ones, col)
        nibbles.append(prev_idx)
    # Pack nibbles into a single integer (MSN = layer 0)
    val = 0
    for nib in nibbles:
        val = (val << 4) | nib
    bits = num_layers * 4
    print(f"        edgelist[{col:2d}] = {bits}'h{val:0{num_layers}x};")

EdgeList (previous 1 per layer, packed as hex):
        edgelist[ 0] = 168'h2bfffdebffffffffdfbfffffffffffffffdfbfffffdd;
        edgelist[ 1] = 168'h2bfffffbffffffffebffffffffffffffffebfffffffe;
        edgelist[ 2] = 168'h2bffffbffffffffffbfffffffffffffffffbffffffff;
        edgelist[ 3] = 168'h002bde00000002bdf000000000000002bdf0000000;
        edgelist[ 4] = 168'h11139f011111139e00111111111111139e00111111;
        edgelist[ 5] = 168'h222210022222222001222222222222222001222222;
        edgelist[ 6] = 168'h333332103333333312033333333333333312033333;
        edgelist[ 7] = 168'h444443204444444420344444444444444420344444;
        edgelist[ 8] = 168'h555554035555555530455555555555555530455555;
        edgelist[ 9] = 168'h666666540666666664506666666666666664506666;
        edgelist[10] = 168'h777777650777777775067777777777777775067777;
        edgelist[11] = 168'h888888706888888886078888888888888886078888;
        edgelist[12] = 168'h999999987099999999780999999999999999780999;
        ed